# 05 – Train the final (deployed) model on all data

All evaluation was done in notebooks 03–04 with a model that never saw the val/test questions or the
11 held-out intents. For the **app** we want a model that knows every intent (e.g. "What is HIV?" was
a held-out intent), so we retrain **once** on **all 2,502 questions of all 109 intents**.

- Same model (PubMedBERT embeddings), loss (MNRL), pairs (q2q + q2a) and settings as notebook 03.
- **Fixed 4 epochs**: there is no data left for early stopping, so we use the best epoch found in the
  experiments (seed study: best epoch 4 in 5 of 6 bi-encoder runs).
- The "I don't know" threshold from notebook 04 (0.854) is reused. It was calibrated on the evaluation
  model, so for the deployed model it is an approximation (stated as a limitation).
- **No new test numbers** come from this notebook: the model has now seen every question, so any score
  on them would be meaningless. The reported results remain those of notebooks 03–04.

**Runtime:** ~12–15 minutes on a Colab T4 GPU.

## 0. Setup (works locally and on Google Colab)

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Eddydev-ALU/ML_Technique_I_Summative_sti-qa-assistant.git"
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers==6.1.0",
                    "transformers==5.20.0", "rank_bm25==0.2.2", "openpyxl==3.1.5"], check=True)
    subprocess.run([sys.executable, "-m", "src.download_data"], check=True)
    subprocess.run([sys.executable, "-m", "src.data_prep"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch

from src.data_prep import load_prepared
from src.evaluate import confidence
from src.models import EmbeddingRetriever, get_device
from src.train import train_bi_encoder_final

print("Device:", get_device(), "| torch", torch.__version__)
if get_device() == "cpu":
    print("WARNING: no GPU found. On Colab: Runtime -> Change runtime type -> T4 GPU, then run again.")

PUBMEDBERT = "NeuML/pubmedbert-base-embeddings"
FINAL_DIR = "models/final_bi_encoder"
EPOCHS = 4
SEED = 42

questions, answer_bank = load_prepared()   # ALL splits: train + val + test + held_out
print(f"{len(questions)} questions, {questions['intent_id'].nunique()} intents, {len(answer_bank)} answers")

## 1. Train on all questions

In [ ]:
start = time.time()
final_model, history = train_bi_encoder_final(PUBMEDBERT, questions, answer_bank, pair_types=("q2q", "q2a"),
                                              epochs=EPOCHS, seed=SEED)
print(f"training time: {time.time() - start:.0f}s")
os.makedirs("results/training_logs", exist_ok=True)
history.to_csv("results/training_logs/final_bi_encoder.csv", index=False)
final_model.save(FINAL_DIR)
print("saved to", FINAL_DIR)

## 2. Sanity checks (NOT an evaluation)
Quick checks that the saved model behaves as expected: questions from the formerly held-out intents
are now answered, and off-topic questions still fall below the threshold.

In [ ]:
with open("results/ood_threshold.json") as f:
    THRESHOLD = json.load(f)["threshold"]

retriever = EmbeddingRetriever(final_model, mode="q2q").fit(questions["question"], questions["intent_id"], answer_bank)
intents = np.array(retriever.intents)

checks = ["What is HIV?", "What is Hepatitis B?", "Where did syphilis come from?",
          "Can I get gonorrhoea from kissing?", "How do I plant maize?", "What causes malaria?"]
s = retriever.score(checks)
for q, c, i in zip(checks, confidence(s), intents[s.argmax(axis=1)]):
    print(f"{'ANSWER ' if c >= THRESHOLD else 'REFUSE '} {c:.3f}  {q:<40} -> {i}")

off_topic = pd.read_csv("evaluation/off_topic_questions.csv")
off_conf = confidence(retriever.score(off_topic["question"].tolist()))
print(f"\nOff-topic questions below the threshold: {(off_conf < THRESHOLD).mean():.0%} of {len(off_topic)}")

## 3. Colab only: download the model
Downloads `final_bi_encoder.zip` (~400 MB). Put it in the project's `data/` folder, like before.
Also save this notebook with its outputs (File → Save a copy in GitHub, or Download .ipynb).

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files

    shutil.make_archive("/content/final_bi_encoder", "zip", root_dir="/content/repo/models", base_dir="final_bi_encoder")
    shutil.copy("results/training_logs/final_bi_encoder.csv", "/content/final_bi_encoder_log.csv")
    files.download("/content/final_bi_encoder.zip")
    files.download("/content/final_bi_encoder_log.csv")
else:
    print("Running locally: the model is in", FINAL_DIR)